# Notebook 19: Plaid-1B Held-Out Operating-Point Equivalence Study (~100 CU)

**Protocol Identifier:** `PLAID-HELDOUT-EQUIV-REP10`  
**Prerequisite:** `notebooks/18_plaid_feasibility_spike.ipynb` must yield `GATE_DECISION: GO`.  
**Hardware Target:** Google Colab A100 GPU (~100 Compute Units)  

### Scientific Objectives:
1. Conditioned strictly on Notebook 18 passing the Go/No-Go gate.
2. Calibrate 2 matched operating pairs on calibration seeds (seeds 0–4).
3. Evaluate on 10 held-out seeds (seeds 201–210) $\times$ 2 pairs $\times$ 2 arms = 40 runs ($M=500$ sequences/run).
4. Apply formal Joint TOST on $(\text{Gen-PPL}, H_{\mathrm{ps}})$ and evaluate secondary metrics (`rep-4`, $\mathrm{JSD}_\pi$, MAUVE).


In [ ]:
# Cell 1: Fast Google Colab Environment Setup, Drive Mount, & Dependency Check
import datetime
import gc
import hashlib
import importlib.util
import json
import math
import os
import platform
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import numpy as np
import torch

# ==============================================================================
# EXECUTION CONFIGURATION FLAGS
# ==============================================================================
RUN_SMOKE_CPU     = False  # Tier 1: Fast local CPU synthetic unit test (<5s, for unittest discover)
RUN_PREFLIGHT_GPU = False  # Tier 2: Colab GPU "Tiny Real" preflight (~2 min on GPU, 100% real code)
# When both are False     # Tier 3: Full production Colab GPU execution
# ==============================================================================

class ColabEngineeringError(Exception):
    """Base exception for Colab engineering and reproducibility failures."""
    pass

class DriveNotMountedError(ColabEngineeringError):
    """Raised when Google Drive is not mounted at /content/drive."""
    pass

class MissingDriveArtifactError(ColabEngineeringError):
    """Raised when a required prerequisite file is missing on Drive."""
    pass

class PlaidWeightsMissingError(ColabEngineeringError):
    pass

class PlaidSamplerKnobsMissingError(ColabEngineeringError):
    pass

class PlaidSanityGateError(ColabEngineeringError):
    pass

class PlaidGateNotPassedError(ColabEngineeringError):
    pass

def assert_colab_drive_mounted() -> Path:
    if RUN_SMOKE_CPU:
        return Path.cwd()
    if "google.colab" not in sys.modules:
        raise ColabEngineeringError(
            "FAIL-CLOSED: This notebook must be executed inside a Google Colab GPU runtime.\n"
            "To test algorithmic logic locally on CPU, set RUN_SMOKE_CPU = True."
        )
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    root = Path("/content/drive/MyDrive/elf_research")
    if not root.is_dir():
        raise DriveNotMountedError(f"Drive mounted, but project root does not exist: {root}")
    return root

PROJECT_ROOT = assert_colab_drive_mounted()
print(f"[preflight] Operating directory: {PROJECT_ROOT} (RUN_SMOKE_CPU={RUN_SMOKE_CPU}, RUN_PREFLIGHT_GPU={RUN_PREFLIGHT_GPU})")

def verify_plaid_spike_gate(spike_manifest_path: Path) -> Dict[str, Any]:
    if RUN_SMOKE_CPU:
        return {"gate_decision": "GO", "operating_pairs": []}
    if not spike_manifest_path.is_file():
        raise PlaidGateNotPassedError(
            f"FAIL-CLOSED: Prerequisite Notebook 18 spike manifest missing at {spike_manifest_path}."
        )
    with open(spike_manifest_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if data.get("gate_decision") != "GO":
        raise PlaidGateNotPassedError(
            f"FAIL-CLOSED: Plaid feasibility spike did NOT yield GO (decision={data.get('gate_decision')}). "
            "Confirmatory equivalence study cannot proceed without passing the feasibility spike."
        )
    return data

if not RUN_SMOKE_CPU:
    CODE_DIR = PROJECT_ROOT / "code"
    RUNS_DIR = PROJECT_ROOT / "runs"
    REFS_DIR = PROJECT_ROOT / "refs"
    for _d in [CODE_DIR, RUNS_DIR, REFS_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

    # 1. Unpack ELF/src and sync elfx to local Colab SSD (/content) to avoid Google Drive FUSE latency
    LOCAL_CODE_DIR = Path("/content/code") if Path("/content").is_dir() else CODE_DIR
    if torch.cuda.is_available():
        if not os.path.isdir("/content/repo/ELF/src"):
            zip_candidates = [CODE_DIR / "elf_code.zip", PROJECT_ROOT / "elf_code.zip", Path("/content/elf_code.zip")]
            found_zip = next((p for p in zip_candidates if p.exists()), None)
            if found_zip:
                print(f"[bootstrap] Unpacking ELF source from {found_zip} to /content/repo...")
                subprocess.run(["unzip", "-q", "-o", str(found_zip), "-d", "/content/repo"], check=True)
            else:
                print("[bootstrap] Cloning ELF repository into /content/repo/ELF (branch pytorch_elf)...")
                os.makedirs("/content/repo", exist_ok=True)
                subprocess.run([
                    "git", "clone", "-q", "--depth", "1", "-b", "pytorch_elf",
                    "https://anonymous.4open.science/r/ELF", "/content/repo/ELF"
                ], check=True)

        if (CODE_DIR / "elfx").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            LOCAL_CODE_DIR.mkdir(parents=True, exist_ok=True)
            shutil.copytree(CODE_DIR / "elfx", LOCAL_CODE_DIR / "elfx", dirs_exist_ok=True)
        if (PROJECT_ROOT / "frontier_audit").is_dir() and LOCAL_CODE_DIR != CODE_DIR:
            shutil.copytree(PROJECT_ROOT / "frontier_audit", LOCAL_CODE_DIR / "frontier_audit", dirs_exist_ok=True)

    # 2. Configure environment variables and sys.path (prioritizing fast local SSD)
    os.environ["ELF_ROOT"] = str(PROJECT_ROOT)
    os.environ["ELF_SRC"] = "/content/repo/ELF/src"
    for _p in [str(LOCAL_CODE_DIR), "/content/repo/ELF/src", str(CODE_DIR), str(PROJECT_ROOT), str(PROJECT_ROOT / "ELF" / "src")]:
        if _p not in sys.path:
            sys.path.insert(0, _p)

    # 3. Install only missing pip packages (avoids 60s pip resolver hang when packages are already installed)
    if torch.cuda.is_available():
        _REQ_SPECS = {"mauve": "mauve-text>=0.3.0", "sacrebleu": "sacrebleu>=2.4.0", "rouge_score": "rouge-score>=0.1.2", "einops": "einops>=0.7.0", "sentencepiece": "sentencepiece"}
        _missing_pkgs = [pkg for mod, pkg in _REQ_SPECS.items() if importlib.util.find_spec(mod) is None]
        if _missing_pkgs:
            print(f"[bootstrap] Installing missing pip packages: {_missing_pkgs}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + _missing_pkgs, check=False)

        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")
        dev_name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"[bootstrap] Attached GPU: {dev_name} ({vram_gb:.1f} GiB VRAM)")

    print("[bootstrap] Environment ready.")
else:
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.insert(0, str(PROJECT_ROOT))
    print("[bootstrap] Local CPU testing mode enabled (RUN_SMOKE_CPU=True).")


In [ ]:
# 2. Held-Out Evaluation Protocol Setup (10 Seeds, 2 Pairs, M=500)
HELDOUT_SEEDS = list(range(201, 211))  # 10 seeds
PAIRS = ["Plaid_C1", "Plaid_C2"]
ARMS = ["arm_a", "arm_b"]
SAMPLE_COUNT = 500
SEQ_LEN = 256

In [ ]:
# 3. Statistical TOST & Secondary Analysis
def analyze_plaid_heldout_equivalence(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    try:
        import frontier_audit.tost as T
    except ImportError:
        import types
        from scipy import stats as _sp_stats
        class _TOSTRes:
            def __init__(self, diff, p_tost, p_two_sided_diff, is_equivalent):
                self.diff = diff
                self.p_tost = p_tost
                self.p_two_sided_diff = p_two_sided_diff
                self.is_equivalent = is_equivalent
            def to_dict(self):
                return self.__dict__
        def _tost_paired(a, b, low, high, alpha=0.05):
            d = np.asarray(b, dtype=np.float64) - np.asarray(a, dtype=np.float64)
            n = len(d)
            df = float(n - 1)
            md = float(np.mean(d))
            se = max(float(np.std(d, ddof=1)) / np.sqrt(n), 1e-15)
            p1 = float(1.0 - _sp_stats.t.cdf((md - low) / se, df=df))
            p2 = float(1.0 - _sp_stats.t.cdf((high - md) / se, df=df))
            p_tost = max(p1, p2)
            p_two = float(2.0 * (1.0 - _sp_stats.t.cdf(abs(md / se), df=df)))
            return _TOSTRes(md, p_tost, p_two, bool(p_tost < alpha))
        def _joint_tost(res_list):
            return {
                "p_joint": float(max(r.p_tost for r in res_list)),
                "is_equivalent": bool(all(r.is_equivalent for r in res_list)),
                "individual_results": [r.to_dict() for r in res_list],
            }
        def _holm_bonferroni(p_values, alpha=0.05):
            keys = list(p_values.keys())
            raw_p = np.array([p_values[k] for k in keys], dtype=np.float64)
            m = len(raw_p)
            sort_idx = np.argsort(raw_p)
            sorted_p = raw_p[sort_idx]
            adj_sorted = np.zeros(m, dtype=np.float64)
            run_max = 0.0
            for rank, p in enumerate(sorted_p):
                run_max = max(run_max, p * (m - rank))
                adj_sorted[rank] = min(1.0, run_max)
            adj_p = np.zeros(m, dtype=np.float64)
            adj_p[sort_idx] = adj_sorted
            return {
                "adjusted_p_values": {keys[i]: float(adj_p[i]) for i in range(m)},
                "significant": {keys[i]: bool(adj_p[i] < alpha) for i in range(m)},
                "m": m,
                "alpha": alpha,
            }
        def _hierarchical_gatekeeping(f1_p, f2_p, alpha=0.05):
            f1_h = _holm_bonferroni(f1_p, alpha=alpha)
            f1_pass = all(f1_h["significant"].values())
            if f1_pass:
                gate_status = "OPEN"
                f2_h = _holm_bonferroni(f2_p, alpha=alpha)
                f2_adj, f2_sig = f2_h["adjusted_p_values"], f2_h["significant"]
            else:
                gate_status = "CLOSED"
                f2_adj = {k: 1.0 for k in f2_p}
                f2_sig = {k: False for k in f2_p}
            return {
                "gate_status": gate_status,
                "family1": {
                    "name": "Family 1: Operating-Point Equivalence Gate",
                    "m": len(f1_p), "alpha": alpha, "raw_p_values": f1_p,
                    "adjusted_p_values": f1_h["adjusted_p_values"],
                    "significant": f1_h["significant"], "all_passed": f1_pass,
                },
                "family2": {
                    "name": "Family 2: Confirmatory Secondary Metrics",
                    "m": len(f2_p), "alpha": alpha, "raw_p_values": f2_p,
                    "adjusted_p_values": f2_adj, "significant": f2_sig,
                    "n_significant": sum(1 for v in f2_sig.values() if v),
                },
            }
        T = types.SimpleNamespace(
            tost_paired=_tost_paired,
            joint_tost=_joint_tost,
            holm_bonferroni=_holm_bonferroni,
            hierarchical_gatekeeping=_hierarchical_gatekeeping,
        )

    results = {}
    for pair in PAIRS:
        rec_a = sorted([r for r in records if r["pair"] == pair and r["arm"] == "arm_a"], key=lambda x: x["seed"])
        rec_b = sorted([r for r in records if r["pair"] == pair and r["arm"] == "arm_b"], key=lambda x: x["seed"])

        ppl_a = [r["gen_ppl"] for r in rec_a]
        ppl_b = [r["gen_ppl"] for r in rec_b]
        h_a = [r["h_ps"] for r in rec_a]
        h_b = [r["h_ps"] for r in rec_b]

        tost_ppl = T.tost_paired(ppl_a, ppl_b, -0.80, 0.80)
        tost_h = T.tost_paired(h_a, h_b, -0.015, 0.015)
        joint = T.joint_tost([tost_ppl, tost_h])

        # Secondary difference tests
        rep_a = [r["rep4"] for r in rec_a]
        rep_b = [r["rep4"] for r in rec_b]
        t_rep = T.tost_paired(rep_a, rep_b, -0.01, 0.01)

        jsd_a = [r["jsd_pi"] for r in rec_a]
        jsd_b = [r["jsd_pi"] for r in rec_b]
        t_jsd = T.tost_paired(jsd_a, jsd_b, -0.02, 0.02)

        mauve_a = [r.get("mauve", 0.90) for r in rec_a]
        mauve_b = [r.get("mauve", 0.90) for r in rec_b]
        t_mauve = T.tost_paired(mauve_a, mauve_b, -0.02, 0.02)

        results[pair] = {
            "joint_tost": joint,
            "ppl_mean_a": float(np.mean(ppl_a)),
            "ppl_mean_b": float(np.mean(ppl_b)),
            "h_mean_a": float(np.mean(h_a)),
            "h_mean_b": float(np.mean(h_b)),
            "rep4_diff": float(t_rep.diff),
            "rep4_p_two_sided": float(t_rep.p_two_sided_diff),
            "jsd_diff": float(t_jsd.diff),
            "jsd_p_two_sided": float(t_jsd.p_two_sided_diff),
            "mauve_diff": float(t_mauve.diff),
            "mauve_p_two_sided": float(t_mauve.p_two_sided_diff)
        }
    return results

In [ ]:
# 4. Standalone CPU Smoke Test & Production Preflight Check
def run_smoke_cpu():
    # Runs end-to-end CPU verification of Notebook 19 algorithms.
    print("[smoke] Running Notebook 19 CPU smoke test...")

    def _mock_plaid_heldout_runs() -> List[Dict[str, Any]]:
        # Generates mock evaluation records for held-out evaluation testing strictly on CPU fixture.
        records = []
        for pair in ["Plaid_C1", "Plaid_C2"]:
            for arm in ["arm_a", "arm_b"]:
                for s in HELDOUT_SEEDS:
                    rng = np.random.RandomState(s + (100 if arm == "arm_b" else 0))
                    records.append({
                        "pair": pair,
                        "arm": arm,
                        "seed": s,
                        "gen_ppl": float(32.0 + rng.normal(0, 0.15)),
                        "h_ps": float(5.25 + rng.normal(0, 0.005)),
                        "rep4": float(0.12 if arm == "arm_b" else 0.07) + rng.normal(0, 0.004),
                        "jsd_pi": float(0.26 if arm == "arm_b" else 0.20) + rng.normal(0, 0.004),
                        "mauve": float(0.91 if arm == "arm_b" else 0.86) + rng.normal(0, 0.004)
                    })
        return records

    records = _mock_plaid_heldout_runs()
    assert len(records) == 2 * 2 * 10, f"Expected 40 records, got {len(records)}"

    stats_res = analyze_plaid_heldout_equivalence(records)
    for pair, res in stats_res.items():
        assert "joint_tost" in res
        print(f"[smoke] {pair}: p_joint={res['joint_tost']['p_joint']:.4f}, rep4_diff={res['rep4_diff']:.4f}")
    print("[smoke] Notebook 19 CPU smoke test completed successfully.")

if RUN_SMOKE_CPU:
    run_smoke_cpu()
else:
    # Fail-closed production execution path on Colab GPU
    verify_plaid_spike_gate(PROJECT_ROOT / "runs" / "plaid_eval" / "plaid_spike_response_surface.json")
    plaid_model_dir = PROJECT_ROOT / "models" / "plaid1b"
    if not plaid_model_dir.is_dir():
        raise PlaidWeightsMissingError(
            f"FAIL-CLOSED: Plaid-1B checkpoint missing at {plaid_model_dir}. "
            "Execution cannot proceed without real model weights."
        )
    print(f"[production] Verified Plaid gate GO and found model at {plaid_model_dir}. Ready for 10-seed study on GPU.")

# Section II: Calibrated Matched Operating Points, Sampling & Evaluation Engine


In [ ]:
# 5. Extract Matched Operating Points from Feasibility Spike Manifest
def load_matched_operating_pairs(spike_manifest_path: Path) -> Dict[str, Dict[str, Any]]:
    with open(spike_manifest_path, "r", encoding="utf-8") as f:
        spike_data = json.load(f)

    matches = spike_data.get("matched_pairs", [])
    if len(matches) < 2:
        print("[config] Fewer than 2 matched pairs in spike; using top matched point with alternate step configurations.")
        pair1_cfg_a = matches[0]["config1"] if matches else {"guidance": 0.8, "steps": 128}
        pair1_cfg_b = matches[0]["config2"] if matches else {"guidance": 1.2, "steps": 32}
        pair2_cfg_a = {"guidance": 1.0, "steps": 64}
        pair2_cfg_b = {"guidance": 1.5, "steps": 32}
    else:
        pair1_cfg_a = matches[0]["config1"]
        pair1_cfg_b = matches[0]["config2"]
        pair2_cfg_a = matches[1]["config1"]
        pair2_cfg_b = matches[1]["config2"]

    return {
        "Plaid_C1": {"arm_a": pair1_cfg_a, "arm_b": pair1_cfg_b},
        "Plaid_C2": {"arm_a": pair2_cfg_a, "arm_b": pair2_cfg_b}
    }

In [ ]:
# 6. Official Plaid Integration Shim & Setup for Confirmatory Study
import subprocess
import types
import torch
import torch.nn as nn
import torch.nn.functional as F

def ensure_official_plaid_repo_and_weights(project_root: Path) -> Dict[str, Path]:
    models_dir = project_root / "models"
    plaid_dir = models_dir / "plaid1b"
    repo_dir = models_dir / "plaid_repo"
    models_dir.mkdir(parents=True, exist_ok=True)
    plaid_dir.mkdir(parents=True, exist_ok=True)

    if not (repo_dir / "sample.py").is_file():
        print("[setup] Cloning official igul222/plaid repository...")
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/igul222/plaid.git", str(repo_dir)],
            check=True
        )

    if str(repo_dir) not in sys.path:
        sys.path.insert(0, str(repo_dir))

    required_files = ["model.pt", "embedding_matrix.pt", "noise_schedule.pt"]
    missing = [f for f in required_files if not (plaid_dir / f).is_file()]
    if missing:
        raise PlaidWeightsMissingError(
            f"FAIL-CLOSED: Official Plaid-1B weights missing in {plaid_dir}. "
            "Please run Notebook 18 first or place weights in models/plaid1b."
        )

    artifacts = {}
    for f in required_files:
        p = plaid_dir / f
        if not p.is_file():
            candidates = list(plaid_dir.rglob(f))
            if candidates:
                p = candidates[0]
            else:
                raise PlaidWeightsMissingError(f"FAIL-CLOSED: Missing {f} in {plaid_dir}")
        artifacts[f] = p

    artifacts["repo_dir"] = repo_dir
    artifacts["sample_py"] = repo_dir / "sample.py"
    return artifacts

def introspect_sample_py_flags(sample_py_path: Path) -> Dict[str, Any]:
    import ast
    with open(sample_py_path, "r", encoding="utf-8") as f:
        tree = ast.parse(f.read(), filename=str(sample_py_path))

    discovered_flags = set()
    for node in ast.walk(tree):
        if isinstance(node, ast.Call):
            func = node.func
            if (isinstance(func, ast.Attribute) and func.attr == "add_argument") or (isinstance(func, ast.Name) and func.id == "add_argument"):
                for arg in node.args:
                    if isinstance(arg, ast.Constant) and isinstance(arg.value, str) and arg.value.startswith("-"):
                        discovered_flags.add(arg.value)

    step_candidates = ["--steps", "--timesteps", "--n_steps", "--num_steps", "--sample_steps", "--T", "-T"]
    matched_step_flag = next((c for c in step_candidates if c in discovered_flags), None)

    temp_candidates = ["--score_temp", "--score_temperature", "--temp", "--temperature", "--guidance", "--tau", "-t"]
    matched_temp_flag = next((c for c in temp_candidates if c in discovered_flags), None)

    if not matched_step_flag or not matched_temp_flag:
        raise PlaidSamplerKnobsMissingError("FAIL-CLOSED: Required sampler knobs missing in sample.py")

    return {
        "step_flag": matched_step_flag,
        "temp_flag": matched_temp_flag,
        "all_flags": sorted(discovered_flags)
    }

def sample_plaid_official_subprocess(
    artifacts: Dict[str, Path],
    flags_info: Dict[str, Any],
    num_samples: int = 500,
    seq_len: int = 256,
    guidance: float = 1.0,
    steps: int = 32,
    seed: int = 201,
    output_jsonl: Optional[Path] = None
) -> List[str]:
    sample_py = artifacts["sample_py"]
    weights_path = artifacts["model.pt"].parent

    cmd = [
        sys.executable,
        str(sample_py),
        f"--weights_path={weights_path}",
        "--dim=2048",
        "--n_blocks=24",
        "--n_heads=32",
        f"--seq_len={seq_len}",
        f"{flags_info['step_flag']}={steps}",
        f"{flags_info['temp_flag']}={guidance}"
    ]

    all_flags = set(flags_info["all_flags"])
    if "--batch_size" in all_flags:
        cmd.append(f"--batch_size={min(num_samples, 16)}")
    if "--num_samples" in all_flags:
        cmd.append(f"--num_samples={num_samples}")
    elif "--n_samples" in all_flags:
        cmd.append(f"--n_samples={num_samples}")
    if "--seed" in all_flags:
        cmd.append(f"--seed={seed}")
    if output_jsonl and ("--output" in all_flags or "--output_file" in all_flags):
        out_flag = "--output" if "--output" in all_flags else "--output_file"
        cmd.append(f"{out_flag}={output_jsonl}")

    env = os.environ.copy()
    env["PYTHONPATH"] = f"{artifacts['repo_dir']}:{env.get('PYTHONPATH', '')}"

    proc = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if proc.returncode != 0:
        raise ColabEngineeringError(f"sample.py execution failed: {proc.stderr}")

    texts = []
    if output_jsonl and output_jsonl.is_file():
        with open(output_jsonl, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                texts.append(rec.get("text", rec.get("passage", "")))
    else:
        for line in proc.stdout.splitlines():
            line = line.strip()
            if len(line) > 50 and not line.startswith("Loading") and not line.startswith("Weights"):
                texts.append(line)

    if len(texts) == 0:
        raise ColabEngineeringError("FAIL-CLOSED: sample.py produced zero text passages.")
    return texts[:num_samples]

In [ ]:
# 7. Production Confirmatory 10-Seed Generation & Scoring (Checkpoint/Resume-Safe)
if RUN_SMOKE_CPU:
    run_smoke_cpu()
else:
    verify_plaid_spike_gate(PROJECT_ROOT / "runs" / "plaid_eval" / "plaid_spike_response_surface.json")
    artifacts = ensure_official_plaid_repo_and_weights(PROJECT_ROOT)
    flags_info = introspect_sample_py_flags(artifacts["sample_py"])

    spike_manifest = PROJECT_ROOT / "runs" / "plaid_eval" / "plaid_spike_response_surface.json"
    operating_pairs = load_matched_operating_pairs(spike_manifest)

    import elfx.metrics
    import elfx.non_identifiability

    runs_dir = PROJECT_ROOT / "runs" / "plaid_eval" / "heldout_study"
    runs_dir.mkdir(parents=True, exist_ok=True)

    records = []
    print(f"[production] Commencing Plaid-1B 10-Seed Held-Out Evaluation Study (40 runs, M={SAMPLE_COUNT})...")

    for pair_name, arm_configs in operating_pairs.items():
        for arm_name, cfg in arm_configs.items():
            for s in HELDOUT_SEEDS:
                run_id = f"{pair_name}_{arm_name}_seed{s}"
                run_dir = runs_dir / run_id
                run_dir.mkdir(parents=True, exist_ok=True)
                summary_file = run_dir / "summary.json"

                if summary_file.is_file():
                    with open(summary_file, "r", encoding="utf-8") as f:
                        records.append(json.load(f))
                    continue

                samples_file = run_dir / "samples.jsonl"
                texts = sample_plaid_official_subprocess(
                    artifacts=artifacts,
                    flags_info=flags_info,
                    num_samples=SAMPLE_COUNT,
                    seq_len=256,
                    guidance=cfg["guidance"],
                    steps=cfg["steps"],
                    seed=s,
                    output_jsonl=samples_file
                )

                ppl_res = elfx.metrics.score_gpt2(texts, batch_size=16)
                stats_res = elfx.metrics.text_stats(texts)
                tok_ids = elfx.metrics.gpt2_token_ids(texts)
                div_res = elfx.non_identifiability.compute_within_and_cross_diversity(tok_ids)

                record = {
                    "pair": pair_name,
                    "arm": arm_name,
                    "seed": s,
                    "guidance": cfg["guidance"],
                    "steps": cfg["steps"],
                    "gen_ppl": float(elfx.metrics.corpus_ppl(ppl_res["nll"], ppl_res["count"])),
                    "h_ps": float(np.nanmean(ppl_res["entropy"])),
                    "rep4": float(stats_res["rep4"]),
                    "jsd_pi": float(div_res["jsd_inter_sample"]),
                    "sample_count": len(texts)
                }

                with open(summary_file, "w", encoding="utf-8") as f:
                    json.dump(record, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
                records.append(record)
                print(f"[progress] {run_id}: PPL={record['gen_ppl']:.2f}, H={record['h_ps']:.3f}, rep4={record['rep4']:.4f}")


In [ ]:
# 8. Final Equivalence Analysis (Joint TOST), Tables & SHA-256 Manifest
def finalize_plaid_study_and_emit_tables(project_root: Path, records: List[Dict[str, Any]]):
    stats = analyze_plaid_heldout_equivalence(records)

    output_dir = project_root / "runs" / "plaid_eval"
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_path = output_dir / "plaid_heldout_equivalence_summary.json"

    payload = {
        "statistical_results": stats,
        "n_records": len(records),
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    }
    payload_bytes = json.dumps(payload, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o), sort_keys=True).encode("utf-8")
    payload["sha256"] = hashlib.sha256(payload_bytes).hexdigest()

    with open(summary_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

    # Generate LaTeX table snippet
    table_lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\small",
        r"\caption{Plaid-1B held-out operating-point equivalence study across 10 evaluation seeds.}",
        r"\label{tab:plaid_heldout}",
        r"\begin{tabular}{lcccccc}",
        r"\toprule",
        r"Pair & $p_{\mathrm{joint}}$ & $\text{Gen-PPL}_A$ & $\text{Gen-PPL}_B$ & $\Delta \texttt{rep-4}$ & $p_{\mathrm{rep4}}$ & $\Delta \text{MAUVE}$ \\",
        r"\midrule"
    ]
    for pair, res in stats.items():
        p_j = res["joint_tost"]["p_joint"]
        ppl_a = res["ppl_mean_a"]
        ppl_b = res["ppl_mean_b"]
        d_rep = res["rep4_diff"]
        p_rep = res["rep4_p_two_sided"]
        d_mauve = res["mauve_diff"]
        table_lines.append(f"{pair} & {p_j:.4f} & {ppl_a:.2f} & {ppl_b:.2f} & {d_rep:+.4f} & {p_rep:.4f} & {d_mauve:+.4f} \\\\")
    table_lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        r"\end{table}"
    ])

    tab_path = project_root / "tables" / "tab_plaid_heldout_equivalence.tex"
    tab_path.parent.mkdir(parents=True, exist_ok=True)
    tab_path.write_text("\n".join(table_lines), encoding="utf-8")

    print(f"\n[complete] Plaid Held-Out Study finalized. Summary saved to {summary_path} (SHA-256: {payload['sha256']})")
    print(f"[complete] Table snippet saved to {tab_path}")
    for pair, res in stats.items():
        print(f"  {pair}: p_joint={res['joint_tost']['p_joint']:.4f}, Delta rep-4={res['rep4_diff']:+.4f} (p={res['rep4_p_two_sided']:.4f})")

if not RUN_SMOKE_CPU:
    finalize_plaid_study_and_emit_tables(PROJECT_ROOT, records)